# 2. Цэвэрлэгээ ба боловсруулалт

Алхам бүрийг `src/clean.py` (шидэлт, тоглолтын явцыг `src/pbp.py`) хийдэг; доорх лог нь `python -m src.clean`-ийн гаралт (бие даалтын Хэсэг 2).

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
P, R, F = ROOT / "data" / "processed", ROOT / "reports", ROOT / "figures"
def show(name):
    display(Image(filename=str(F / f"{name}.png"), width=760))

In [2]:
log = pd.read_csv(R / "cleaning_log.csv"); log

,step,description,rows_before,rows_after,note
0,C7,Давхардлыг хасав (game_id),252,252,NaN
1,C1,home_name: багийн нэрийг стандарт team_key руу...,252,252,16 түүхий нэр → 14 баг
2,C1,away_name: багийн нэрийг стандарт team_key руу...,252,252,16 түүхий нэр → 14 баг
3,C1b,Газрын нэрийг стандартчлав,252,252,6 хувилбар → 2
4,C11,Тоглолтын үе шатыг (regular/playoff) хуваарийн...,252,252,playoff: 37
5,C9,games: өгөгдлийн төрлийг тохируулав,252,252,NaN
6,C1,team_name: багийн нэрийг стандарт team_key руу...,504,504,16 түүхий нэр → 14 баг
7,C5,"'м-о' хэлбэрийн FG, TP, FT баганыг made/attemp...",504,504,NaN
8,C7,"Давхардлыг хасав (game_id, team_key)",504,504,NaN
9,C13,"Гарчгийн оноо, box score, quarter-уудын нийлбэ...",252,252,"гарчиг≠box: 3, гарчиг≠quarter: 6, ялагч өөрчлө..."


## Жишээ: өмнө / дараа

In [3]:
raw = pd.read_csv(ROOT / "data" / "raw" / "players_season_raw.csv")
print("Багийн түүхий нэрс:", sorted(raw.team_name.unique()))
ps = pd.read_csv(P / "players_season.csv")
print("Стандарт баг:", sorted(ps.team_key.unique()))

Багийн түүхий нэрс: ['BCH Knights', 'BISHRELT METAL', 'BUREN SCORE MONGOLIANS', 'Darkhan United', 'Khan Falcons', 'OMNI ERDENET MINERS', 'Omni Erdenet Miners', 'SELENGE BODONS', 'SG Apes', 'SUA TLG Tushig Bagana', 'Selenge Bodons', 'Sono Brothers', 'South Gobi Broncos', 'Western Warriors', 'XAC BRONCOS', 'ZAVKHAN BROTHERS']
Стандарт баг: ['bch_knights', 'bishrelt_metal', 'burenscore_mongolians', 'darkhan_united', 'erdenet_miners', 'khan_falcons', 'khasin_khuleguud', 'selenge_bodons', 'sg_apes', 'sono_brothers', 'south_gobi_broncos', 'sua_tlg_tushig_bagana', 'western_warriors', 'zavkhan_brothers']


In [4]:
pg_raw = pd.read_csv(ROOT / "data" / "raw" / "player_games_raw.csv")
pg = pd.read_csv(P / "player_games.csv")
display(pg_raw[["name", "FG", "TP", "FT", "dnp"]].head(3)); display(pg[["player_id", "FGM", "FGA", "TPM", "TPA", "FTM", "FTA"]].head(3))

,name,FG,TP,FT,dnp
0,ALTANGEREL Azbayar,7-9,0-0,5-6,False
1,MILLER Ian,4-18,3-11,2-4,False
2,BAATAR-ERDENE Barsbold,5-13,1-8,2-2,False


,player_id,FGM,FGA,TPM,TPA,FTM,FTA
0,altangerel-azbayar,7,9,0,0,5,6
1,miller-ian,4,18,3,11,2,4
2,baatar-erdene-barsbold,5,13,1,8,2,2


In [5]:
prof = pd.read_csv(ROOT / "data" / "raw" / "players_profile_raw.csv")
prof[prof.birth_year.isin([1900, 2025])][["player_slug", "birth_year", "height_cm"]]

,player_slug,birth_year,height_cm
190,rodrigue-andela,1900.0,202.0
209,chinges-temuulen,2025.0,195.0
213,pok-simba,1900.0,210.0
218,manhom-lino,1900.0,NaN


## Дутуу утга, нэрийн тааруулалт, онооны тулгалт

In [6]:
display(pd.read_csv(R / "missing.csv"))
m = pd.read_csv(R / "name_matches.csv"); display(m[m.left_id != m.right_id].head(10))
v = pd.read_csv(R / "ppg_validation.csv"); v[v.flag]

,column,n_missing,pct_missing
0,position,0,0.00
1,height_cm,23,7.23
2,birth_year,15,4.72
3,FT_pct,38,11.95
4,TP_pct,34,10.69
5,FGA,0,0.00


,left_id,left_name,right_id,right_name,score,same_team,n_candidates
0,byron-wesley-2,BYRON Wesley,byron-wesley,BYRON Wesley,1.00,False,1
1,scrubb-jay,SCRUBB Jay,scrubb-jayden-amari,SCRUBB Jayden Amari,0.95,True,1
3,gankhuyag-dulguun-2,GANKHUYAG Dulguun,gankhuyag-dulguun,GANKHUYAG Dulguun,1.00,True,1
6,ochirbat-erdenetsogt-2,OCHIRBAT Erdenetsogt,ochirbat-erdenetsogt,OCHIRBAT Erdenetsogt,1.00,True,1
10,azzayaa-batkhuyag-2,AZZAYAA Batkhuyag,azzayaa-batkhuyag,AZZAYAA Batkhuyag,1.00,True,1
14,tolbert-jordan,TOLBERT Jordan,tolbert-jordan-wayne,TOLBERT Jordan Wayne,0.95,False,1
15,otgonbayar-usukhbayar-2,OTGONBAYAR Usukhbayar,otgonbayar-usukhbayar,OTGONBAYAR Usukhbayar,1.00,False,1
16,norvel-pelle,NORVEL Pelle,norvel-pelle-2,NORVEL Pelle,1.00,False,1
19,elbeg-tushig-2,ELBEG Tushig,elbeg-tushig,ELBEG Tushig,1.00,False,1
20,enkhtaivan-iderjavkhlan-3,ENKHTAIVAN Iderjavkhlan,enkhtaivan-iderjavkhlan-2,ENKHTAIVAN Iderjavkhlan,1.00,True,1


,player_id,season,team_key,PPG,GP,PTS_total,FGM,FGA,TPM,TPA,FTM,FTA,GP_games,traded,gp_games,ppg_games,diff,flag


## Шидэлт ба тоглолтын явц (C14–C16)

- **C14:** шидэлтийг (тоглолт, тал, өмсгөлийн дугаар)-аар тоглогчтой холбож, пикселийг метр болгов. Хувиргалтыг MBA сайтын өөрийнх нь шидэлтийн зургийн кодоос авсан: талбай 28 × 15 м, 1 м = 22.75 пиксел. Баг бүрийн шидэлтийг нэг сагс руу эргүүлж нэгтгээд 5 бүсэд хуваав.
Эхний сарын (2025 оны 10-р сар) 29 тоглолтод байршил бүртгэгдээгүй — 2 онооны бүх шидэлт сагснаас хол, цөөн тогтмол цэг дээр байгаа тул эдгээрийг байршлын шинжилгээнээс хассан (доорх хүснэгт).
- **C15:** тоглолтын явцын цагийг секунд, онооны зөрүү болгов.
- **C16:** оновчтой дамжуулалтыг ойролцоох оносон шидэлттэй холбов.

In [7]:
shots = pd.read_csv(P / "shots.csv")
print(f"Шидэлт: {len(shots):,}; хайрцаг онооны нийт шидэлт: {int(pg.FGA.sum()):,} (тэнцүү байх ёстой)")
print(f"Байршлын шинжилгээнээс хассан: {(~shots.location_ok).mean():.1%} (байршил бүртгэгдээгүй тоглолт + 2/3 онооны шошготой зөрсөн)")
shots.head()

Шидэлт: 33,347; хайрцаг онооны нийт шидэлт: 33,347 (тэнцүү байх ёстой)
Байршлын шинжилгээнээс хассан: 12.6% (байршил бүртгэгдээгүй тоглолт + 2/3 онооны шошготой зөрсөн)


,game_id,seq,side,number,period,made,three,player_id,team_key,is_import,court_x,court_y,dist,zone,location_ok
0,223,19,away,4,1,True,False,tumenbayar-uuganbayar,bishrelt_metal,False,4.18,1.92,6.16,Дунд зай,False
1,223,22,home,9,1,False,True,chinges-temuulen,khasin_khuleguud,False,1.19,0.77,6.74,Булангийн 3,False
2,223,24,away,13,1,False,False,erdenetsetseg-margad,bishrelt_metal,False,4.04,2.18,5.86,Дунд зай,False
3,223,26,home,5,1,True,False,baatar-erdene-barsbold,khasin_khuleguud,False,0.26,2.27,5.39,Дунд зай,False
4,223,33,away,3,1,True,False,bradford-christian,bishrelt_metal,True,3.52,1.04,6.75,Дунд зай,False


In [8]:
gm = pd.read_csv(P / "games.csv")[["game_id", "date"]]
chk = shots.merge(gm, on="game_id").assign(сар=lambda d: d.date.str[:7], ойр=lambda d: ~d.three & (d.dist <= 3))
chk.groupby("сар").agg(тоглолт=("game_id", "nunique"), шидэлт=("dist", "size"), гурван_метр_доторх_хувь=("ойр", "mean"), байршил_зөв=("location_ok", "mean")).round(3)

,тоглолт,шидэлт,гурван_метр_доторх_хувь,байршил_зөв
сар,,,,
2025-10,29,3797,0.000,0.000
2025-11,26,3457,0.458,0.999
2025-12,23,3017,0.452,0.991
2026-01,35,4562,0.422,0.978
2026-02,24,3113,0.443,0.983
2026-03,30,3875,0.453,0.975
2026-04,5,689,0.425,0.984
2026-10,56,7609,0.434,0.991
2026-11,24,3228,0.444,0.989


In [9]:
e = pd.read_csv(P / "pbp.csv.gz"); g = pd.read_csv(P / "games.csv").set_index("game_id")
last = e.sort_values(["game_id", "seq"]).groupby("game_id")[["home", "away"]].last()
ok = (last.home == g.home_pts.reindex(last.index)) & (last.away == g.away_pts.reindex(last.index))
print(f"Тоглолтын явцын эцсийн оноо гарчигтай таарсан: {ok.sum()}/{len(ok)}; зөрсөн:", sorted(ok.index[~ok]))

Тоглолтын явцын эцсийн оноо гарчигтай таарсан: 246/252; зөрсөн: [151, 162, 203, 217, 240, 263]


## Шинэ үзүүлэлтүүд

`age`, `pos_group`, 36 минутад ногдох (`pts36` …), шидэлтийн оновч (`ts_pct`), 3 оноогоор шидсэн хувь (`tpa_rate`), торгууль авалт (`ftr`), `scorer_tier`, `is_national`, `region`; багийн тоглолтод ялалтын 4 хүчин зүйл (`efg`, `tov_pct`, `orb_pct`, `ftr`), 100 довтолгоонд ногдох оноо (`ortg`, `drtg`, `net`); тоглолтод багийн хүчний оноо (Elo); шидэлтэд зай (м) ба бүс.

In [10]:
ps[["player_name", "season", "PPG", "pts36", "ts_pct", "tpa_rate", "scorer_tier", "pos_group", "age", "region", "is_national"]].head()

,player_name,season,PPG,pts36,ts_pct,tpa_rate,scorer_tier,pos_group,age,region,is_national
0,BYRON Wesley,2025-2026,27.9,30.904615,0.590822,0.250518,Од,Forward,33.0,Улаанбаатар,False
1,SAINJARGAL Khishigbayar,2025-2026,1.0,6.206897,0.333333,0.000000,Бага,Center,NaN,Орон нутаг,False
2,UORREN Davion,2025-2026,26.7,31.006452,0.582254,0.263485,Од,Forward,26.0,Улаанбаатар,False
3,SAINJARGAL Khishigbayar,2025-2026,0.0,0.000000,NaN,NaN,Бага,Center,25.0,Орон нутаг,False
4,SCRUBB Jay,2025-2026,25.2,27.913846,0.579877,0.490566,Од,Guard,25.0,Орон нутаг,False


In [11]:
notes = log.drop_duplicates("step").set_index("step").note.fillna("")
for step in ["C1", "C10", "C12", "C13"]:
    print(step, "—", notes.get(step, ""))

C1 — 16 түүхий нэр → 14 баг
C10 — бүх тоглолтоор 77 зөрүүтэй → MBA-ийн хүснэгт зөвхөн regular season-ийг тооцдог нь тогтоогдов; regular-аар, баг сольсон тоглогчийг бүх багаар нэгтгэхэд 0 зөрүүтэй
C12 — төрсөн он: 4, өндөр: 0; нийт 4
C13 — гарчиг≠box: 3, гарчиг≠quarter: 6, ялагч өөрчлөгдөх: 0


**[Ноорог — өөрийн үгээр дахин бичих]** Дээрх 4 алхмаас хамгийн чухал олдворыг (жишээ нь C10: сайтын нэгтгэл зөвхөн үндсэн шатыг тооцдог) өөрийн үгээр тайлбарлана.